# Atividade
## Normalização, escalonamento e codificação de dados categóricos
Ciência de Dados · Bacharelado em Engenharia de Software · UNIFSA


### Contexto
Uma operadora de telecomunicações deseja compreender melhor sua base de clientes. A diretoria comercial solicitou à equipe de dados dois produtos:

1. um **índice de valor do cliente**, que permita ranquear os clientes combinando tempo de relacionamento e faturamento;
2. uma **análise do cancelamento (churn)**, que indique quais características dos clientes estão associadas a uma maior taxa de cancelamento.

As métricas disponíveis estão em escalas diferentes e grande parte das informações está em formato textual. Antes de qualquer análise, é necessário preparar os dados.

### Base de dados
IBM. *Telco Customer Churn* (base de exemplo). 7.043 clientes e 21 colunas. Endereço público para carga:

https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv



### Dicionário de dados (colunas utilizadas)
| Coluna | Descrição |
|---|---|
| customerID | identificador do cliente |
| gender | sexo do titular (Male, Female) |
| Partner, Dependents | possui cônjuge; possui dependentes (Yes, No) |
| tenure | tempo de relacionamento, em meses |
| PhoneService, PaperlessBilling | possui telefone; usa fatura digital (Yes, No) |
| MultipleLines | possui múltiplas linhas (Yes, No, No phone service) |
| InternetService | tipo de internet (DSL, Fiber optic, No) |
| OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | serviços adicionais (Yes, No, No internet service) |
| Contract | tipo de contrato (Month-to-month, One year, Two year) |
| PaymentMethod | forma de pagamento (quatro categorias) |
| MonthlyCharges | valor da mensalidade |
| TotalCharges | valor total faturado desde o início do contrato |
| Churn | o cliente cancelou o serviço no último mês (Yes, No) |

---
## Parte 1 — Carga e preparação dos dados

**1.1** Carregue a base a partir do endereço público informado no início do notebook. Exiba o número de linhas e de colunas e os tipos de cada coluna.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Número de linhas e colunas:", df.shape)
print("\nTipos de cada coluna:")
print(df.dtypes)

**1.2** A coluna `TotalCharges` deveria ser numérica, mas foi lida como texto. Converta-a para numérico e identifique os registros que não puderam ser convertidos. Examine o valor de `tenure` nesses registros.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
registros_invalidos = df[df["TotalCharges"].isna()]
print("Quantidade de registros não convertidos:", len(registros_invalidos))
print("Valores de tenure nesses registros:", registros_invalidos["tenure"].unique())
registros_invalidos[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

**1.3** Com base no que foi observado em 1.2, decida como tratar esses registros e implemente a decisão. Justifique a escolha considerando o significado do negócio (o que representa um cliente com esse valor de `tenure`?).

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("Total de valores ausentes em TotalCharges:", df["TotalCharges"].isna().sum())

**Resposta:** Optou-se por preencher os 11 registros ausentes de `TotalCharges` com `0` (`fillna(0)`). Todos esses registros possuem `tenure = 0`, o que no contexto do negócio representa clientes recém-contratados que ainda estão no primeiro mês de vigência do contrato e, portanto, ainda não tiveram nenhuma fatura mensal fechada no faturamento acumulado. Atribuir `0` reflete fielmente o valor total faturado até o momento sem descartar clientes ativos da base.

---
## Parte 2 — Escalonamento e índice de valor do cliente

**2.1** Apresente as estatísticas descritivas de `tenure`, `MonthlyCharges` e `TotalCharges`. Compare as amplitudes (mínimo e máximo) das três métricas.

In [ ]:
metricas = ["tenure", "MonthlyCharges", "TotalCharges"]
desc = df[metricas].describe()
desc.loc["amplitude"] = desc.loc["max"] - desc.loc["min"]
desc

**Resposta:** As três métricas apresentam amplitudes e escalas muito distintas:
- **`tenure`**: mínimo de `0,00` e máximo de `72,00` meses (amplitude de `72,00`, média de `32,37` e desvio-padrão de `24,56`).
- **`MonthlyCharges`**: mínimo de `18,25` e máximo de `118,75` (amplitude de `100,50`, média de `64,76` e desvio-padrão de `30,09`).
- **`TotalCharges`**: mínimo de `0,00` e máximo de `8.684,80` (amplitude de `8.684,80`, média de `2.279,73` e desvio-padrão de `2.266,79`).

A amplitude de `TotalCharges` (`8.684,80`) é cerca de 120 vezes maior que a de `tenure` (`72,00`) e cerca de 86 vezes maior que a de `MonthlyCharges` (`100,50`).

**2.2** Construa um **índice bruto** somando diretamente as três métricas. Calcule a correlação do índice bruto com cada uma delas. Qual métrica determina o índice? Por quê?

In [ ]:
df["indice_bruto"] = df["tenure"] + df["MonthlyCharges"] + df["TotalCharges"]
df[metricas].corrwith(df["indice_bruto"])

**Resposta:** A métrica que determina o índice bruto é `TotalCharges`, cuja correlação com o índice é de `0,999973` (praticamente `1,00`), enquanto `tenure` (`0,8258`) e `MonthlyCharges` (`0,6556`) apresentam correlações que apenas refletem a relação delas próprias com `TotalCharges`. Isso ocorre porque `TotalCharges` está em uma escala numérica e de variabilidade (desvio-padrão de `2.266,79`) ordens de grandeza superior às de `tenure` (`24,56`) e `MonthlyCharges` (`30,09`), dominando quase integralmente a soma direta.

**2.3** Padronize as três métricas com **z-score** e construa o **índice padronizado** como a soma das três métricas padronizadas. Repita o cálculo de correlações do item 2.2.

In [ ]:
df_z = (df[metricas] - df[metricas].mean()) / df[metricas].std()
df["indice_padronizado"] = df_z.sum(axis=1)
df[metricas].corrwith(df["indice_padronizado"])

**2.4** Liste os 10 clientes com maior índice bruto e os 10 com maior índice padronizado. Quantos clientes aparecem nas duas listas? O que mudou no perfil dos clientes mais bem posicionados?

In [ ]:
top10_bruto = df.nlargest(10, "indice_bruto")[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "indice_bruto"]]
top10_padronizado = df.nlargest(10, "indice_padronizado")[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "indice_padronizado"]]

print("Top 10 - Índice Bruto:")
print(top10_bruto.to_string(index=False))

print("\nTop 10 - Índice Padronizado:")
print(top10_padronizado.to_string(index=False))

em_comum = set(top10_bruto["customerID"]).intersection(set(top10_padronizado["customerID"]))
print("\nQuantidade de clientes nas duas listas:", len(em_comum))
print("Clientes em comum:", sorted(em_comum))

**Resposta:** **8 clientes** aparecem nas duas listas (`0675-NCDYU`, `2889-FPWRM`, `6650-BWFRT`, `7569-NMZYQ`, `8984-HPEMB`, `9739-JLPQJ`, `9788-HNGUT` e `9924-JPRMC`). Com a padronização, o ranking deixou de ordenar os clientes exclusivamente pelo faturamento acumulado (`TotalCharges`) e passou a valorizar de forma equilibrada a mensalidade (`MonthlyCharges`) e o tempo máximo de contrato (`tenure = 72`). Por exemplo, o cliente `7569-NMZYQ` (que possui a maior mensalidade da base, `118,75`, e `tenure = 72`) assumiu o 1º lugar no índice padronizado, superando `2889-FPWRM` (`MonthlyCharges = 117,80`), e clientes com mensalidades mais altas e `tenure = 72` (`1488-PBLJN` e `6904-JLBGY`) entraram no Top 10 no lugar de `8879-XUAHX` (`tenure = 71`) e `0164-APGRB` (`MonthlyCharges = 114,90`).

**2.5** Aplique o **min-max** às três métricas e construa um terceiro índice. Compare a ordenação dos clientes no índice min-max com a do índice padronizado (sugestão: correlação entre os dois índices).

In [ ]:
df_minmax = (df[metricas] - df[metricas].min()) / (df[metricas].max() - df[metricas].min())
df["indice_minmax"] = df_minmax.sum(axis=1)

print("Correlação de Pearson:", df["indice_padronizado"].corr(df["indice_minmax"]))
print("Correlação de Spearman (ordenação):", df["indice_padronizado"].corr(df["indice_minmax"], method="spearman"))

**Resposta:** A ordenação dos clientes nos dois índices é praticamente idêntica, apresentando correlação linear de Pearson de `0,9995` e correlação de postos de Spearman de `0,9994` (além de os mesmos 10 clientes comporem o Top 10 de ambos os índices). Isso ocorre porque nenhuma das três variáveis possui *outliers* extremos capazes de distorcer a amplitude em relação ao desvio-padrão, fazendo com que tanto o escalonamento min-max quanto o z-score atribuam pesos relativos muito semelhantes às três métricas.

**2.6** Calcule a matriz de correlação entre `tenure`, `MonthlyCharges` e `TotalCharges`. Considerando que o faturamento total resulta, aproximadamente, do tempo de relacionamento multiplicado pela mensalidade, discuta se faz sentido manter as três métricas no índice. Proponha e implemente a versão final do índice de valor do cliente.

In [ ]:
print("Matriz de correlação:")
print(df[metricas].corr())

df["indice_valor_cliente"] = df_z["tenure"] + df_z["MonthlyCharges"]
print("\nCorrelação do índice final com as métricas:")
print(df[metricas].corrwith(df["indice_valor_cliente"]))

**Resposta:** A matriz de correlação mostra que `TotalCharges` possui forte correlação tanto com `tenure` (`0,8262`) quanto com `MonthlyCharges` (`0,6512`), enquanto `tenure` e `MonthlyCharges` têm correlação baixa entre si (`0,2479`). Como `TotalCharges` é aproximadamente o produto de `tenure` por `MonthlyCharges`, mantê-la no índice juntamente com as outras duas variáveis gera **redundância (dupla contagem)** do tempo de contrato e da mensalidade. Portanto, a versão final proposta para o índice de valor do cliente é a soma das variáveis padronizadas de tempo de relacionamento e mensalidade (`z_tenure + z_MonthlyCharges`), que equilibra exatamente o peso de ambas as dimensões (correlação de `0,7899` com `tenure` e `0,7899` com `MonthlyCharges`, além de `0,9351` com `TotalCharges`).

---
## Parte 3 — Forma da distribuição

**3.1** Construa os histogramas das três métricas e calcule a assimetria de cada uma (`.skew()`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(metricas):
    axes[i].hist(df[col], bins=30, edgecolor="black")
    axes[i].set_title(f"{col} (skew = {df[col].skew():.4f})")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequência")
plt.tight_layout()
plt.show()

df[metricas].skew()

**3.2** Aplique a transformação `np.log1p` à coluna `TotalCharges`. Compare o histograma e a assimetria antes e depois da transformação. A transformação logarítmica é adequada para esta coluna? Justifique com base nos valores obtidos.

In [ ]:
total_charges_log = np.log1p(df["TotalCharges"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["TotalCharges"], bins=30, edgecolor="black")
axes[0].set_title(f"Original (skew = {df['TotalCharges'].skew():.4f})")
axes[0].set_xlabel("TotalCharges")
axes[0].set_ylabel("Frequência")

axes[1].hist(total_charges_log, bins=30, edgecolor="black")
axes[1].set_title(f"Log1p (skew = {total_charges_log.skew():.4f})")
axes[1].set_xlabel("np.log1p(TotalCharges)")
axes[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

print("Assimetria antes (original):", df["TotalCharges"].skew())
print("Assimetria depois (log1p):", total_charges_log.skew())

**Resposta:** Antes da transformação, `TotalCharges` apresenta assimetria positiva moderada à direita (`skew = 0,9632`). Após a aplicação de `np.log1p`, a assimetria passa a ser negativa (`skew = -0,8245`), deslocando a cauda para a esquerda. Embora a magnitude absoluta da assimetria tenha diminuído levemente (de `0,9632` para `0,8245`), a transformação logarítmica **não é ideal (ou mostra-se pouco adequada)** para simetrizar esta coluna: como a assimetria original já era moderada (`< 1,0`) e a variável é limitada superiormente (sem *outliers* extremos) com presença de valores nulos/baixos, a função logarítmica comprime em excesso a faixa superior e alonga a cauda inferior, apenas invertendo o sinal da assimetria em vez de aproximá-la de zero.

---
## Parte 4 — Codificação das variáveis categóricas

**4.1** Classifique as variáveis abaixo e indique a técnica de codificação adequada, preenchendo a tabela.

| Variável | Tipo (nominal, ordinal ou binária) | Técnica de codificação |
|---|---|---|
| gender | Binária | Codificação binária (0 e 1) |
| Partner | Binária | Codificação binária (0 e 1) |
| PhoneService | Binária | Codificação binária (0 e 1) |
| MultipleLines | Nominal (binária após consolidar "No phone service" com "No") | Codificação binária (0 e 1) |
| InternetService | Nominal | One-Hot Encoding |
| OnlineSecurity | Nominal (binária após consolidar "No internet service" com "No") | Codificação binária (0 e 1) |
| Contract | Ordinal | Codificação ordinal (0, 1 e 2) |
| PaymentMethod | Nominal | One-Hot Encoding |
| Churn | Binária | Codificação binária (0 e 1) |

**4.2** Converta para 0 e 1 as variáveis binárias `gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling` e `Churn`.

In [ ]:
df["gender"] = df["gender"].map({"Female": 0, "Male": 1})
colunas_binarias = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
for col in colunas_binarias:
    df[col] = df[col].map({"No": 0, "Yes": 1})

df[["gender"] + colunas_binarias].head()

**4.3** As colunas de serviços adicionais (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) e a coluna `MultipleLines` possuem três valores. Analise o significado de "No internet service" e "No phone service", decida como codificar essas colunas e implemente a decisão.

In [ ]:
colunas_servicos = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "MultipleLines"
]
for col in colunas_servicos:
    df[col] = df[col].map({"Yes": 1, "No": 0, "No internet service": 0, "No phone service": 0})

df[colunas_servicos].head()

**Resposta:** As categorias `"No internet service"` e `"No phone service"` indicam que o cliente não contratou o serviço base (internet ou telefonia) e, por consequência lógica, **não possui** o serviço adicional avaliado. Como a ausência de internet ou telefone já está representada nas variáveis `InternetService` e `PhoneService`, criar colunas *dummy* para essas categorias apenas introduziria redundância perfeita (multicolinearidade). Assim, a decisão adequada é mapeá-las para `0` (junto com `"No"`, indicando ausência do serviço adicional) e atribuir `1` apenas para `"Yes"`.

**4.4** Codifique `Contract` de forma ordinal, respeitando a ordem de duração do contrato.

In [ ]:
df["Contract"] = df["Contract"].map({"Month-to-month": 0, "One year": 1, "Two year": 2})
df["Contract"].value_counts().sort_index()

**4.5** Aplique one-hot encoding a `InternetService` e `PaymentMethod`. Utilize ou não o parâmetro `drop_first` e justifique a escolha.

In [ ]:
df = pd.get_dummies(df, columns=["InternetService", "PaymentMethod"], drop_first=False, dtype=int)
colunas_dummies = [c for c in df.columns if c.startswith("InternetService_") or c.startswith("PaymentMethod_")]
df[colunas_dummies].head()

**Resposta:** Optou-se por `drop_first=False` porque o objetivo na Parte 5 é realizar uma análise exploratória de taxas de cancelamento e de correlação de cada categoria individual com `Churn`. Manter todas as colunas *dummy* permite inspecionar diretamente o efeito e a correlação de todas as modalidades de internet (`DSL`, `Fiber optic` e `No`) e de todas as formas de pagamento sem ocultar nenhuma categoria de referência (o uso de `drop_first=True` seria necessário apenas ao ajustar modelos lineares sem regularização, para evitar multicolinearidade perfeita).

---
## Parte 5 — Análise do cancelamento para a diretoria comercial

**5.1** Utilizando a coluna `Churn` codificada, calcule a taxa de cancelamento (proporção de clientes que cancelaram) por tipo de contrato, por forma de pagamento e por tipo de internet.

In [ ]:
print("Taxa de cancelamento por tipo de contrato (0=Month-to-month, 1=One year, 2=Two year):")
print(df.groupby("Contract")["Churn"].mean())

cols_pagamento = [c for c in df.columns if c.startswith("PaymentMethod_")]
print("\nTaxa de cancelamento por forma de pagamento:")
print(pd.Series({c.replace("PaymentMethod_", ""): df.loc[df[c] == 1, "Churn"].mean() for c in cols_pagamento}).sort_values(ascending=False))

cols_internet = [c for c in df.columns if c.startswith("InternetService_")]
print("\nTaxa de cancelamento por tipo de internet:")
print(pd.Series({c.replace("InternetService_", ""): df.loc[df[c] == 1, "Churn"].mean() for c in cols_internet}).sort_values(ascending=False))

**5.2** Calcule a correlação entre `Churn` e as demais variáveis codificadas e numéricas. Apresente as cinco variáveis com maior correlação em valor absoluto, indicando o sentido (positivo ou negativo) de cada uma.

In [ ]:
colunas_analise = [
    c for c in df.columns
    if c not in ["customerID", "Churn", "indice_bruto", "indice_padronizado", "indice_minmax", "indice_valor_cliente"]
]
correlacoes = df[colunas_analise].corrwith(df["Churn"])
top5 = correlacoes.reindex(correlacoes.abs().sort_values(ascending=False).index).head(5)

resultado_top5 = pd.DataFrame({
    "Correlação": top5,
    "Valor Absoluto": top5.abs(),
    "Sentido": np.where(top5 > 0, "Positivo", "Negativo")
})
resultado_top5

**5.3** Redija um parágrafo de até oito linhas dirigido à diretoria comercial, com uma recomendação para reduzir o cancelamento. A recomendação deve estar apoiada nos números obtidos em 5.1 e 5.2.

**Resposta:** À Diretoria Comercial: A análise revela que o cancelamento está fortemente concentrado nos clientes de contrato mensal (*Month-to-month*), cuja taxa de churn atinge `42,71%`, contra `11,27%` nos contratos anuais e apenas `2,83%` nos bienais (`Contract` tem a maior correlação com `Churn`, $r = -0,3967$, sentido negativo). Além disso, o risco é maior nos primeiros meses de relacionamento (`tenure`, $r = -0,3522$), entre usuários de fibra óptica (`InternetService_Fiber optic`, $r = +0,3080$, com `41,89%` de churn ante `18,96%` no DSL e `7,40%` sem internet) e entre os que pagam por cheque eletrônico (`PaymentMethod_Electronic check`, $r = +0,3019$, com `45,29%` de churn contra `15,24%` no cartão de crédito automático e `16,71%` na transferência automática). Recomenda-se implementar imediatamente três ações conjuntas: (1) oferecer descontos progressivos para migrar clientes do contrato mensal para planos de 1 ou 2 anos; (2) conceder incentivos na fatura para migração do cheque eletrônico para pagamento automático (cartão ou débito em conta); e (3) criar uma régua de acompanhamento técnico e de qualidade nos primeiros meses dos clientes de fibra óptica.

---
## Parte 6 — Questões discursivas

**6.1** Explique por que a codificação por rótulo (atribuir os números 0, 1, 2 e 3 às categorias de `PaymentMethod`) seria inadequada para a análise realizada na Parte 5.

**Resposta:** `PaymentMethod` é uma variável categórica **nominal**, ou seja, suas quatro categorias não possuem relação de ordem ou hierarquia entre si. Atribuir os rótulos `0, 1, 2 e 3` imporia uma ordenação e distâncias numéricas artificiais entre as formas de pagamento (tratando a categoria `3` como superior à `1` e supondo intervalos iguais entre categorias adjacentes). Na Parte 5, isso invalidaria o cálculo da correlação de Pearson com `Churn` — cujo valor e sinal mudariam arbitrariamente conforme a ordem escolhida para numerar as categorias — e impediria identificar isoladamente que `Electronic check` ($r = +0,3019$) é a modalidade fortemente associada ao cancelamento.

**6.2** Descreva uma análise sobre esta base em que o escalonamento das métricas numéricas não alteraria o resultado. Justifique.

**Resposta:** O cálculo da **correlação linear de Pearson (ou de postos de Spearman)** entre cada métrica numérica individual (`tenure`, `MonthlyCharges` ou `TotalCharges`) e o `Churn` (como realizado no item 5.2), ou o ajuste de uma **árvore de decisão (ou Random Forest)** para prever `Churn`, não teria seu resultado alterado pelo escalonamento. Isso ocorre porque tanto o z-score quanto o min-max são transformações lineares estritamente crescentes ($x' = ax + b$, com $a > 0$), que preservam a ordem dos dados, os pontos de corte baseados em impureza (Gini/entropia) e a correlação individual de cada variável com o alvo.

**6.3** O coeficiente de correlação entre `tenure` e `Churn` muda se `tenure` for padronizada por z-score ou escalonada por min-max? Justifique matematicamente e, em seguida, verifique no código.

In [ ]:
tenure_z = (df["tenure"] - df["tenure"].mean()) / df["tenure"].std()
tenure_minmax = (df["tenure"] - df["tenure"].min()) / (df["tenure"].max() - df["tenure"].min())

print("Correlação original (tenure x Churn):       ", df["tenure"].corr(df["Churn"]))
print("Correlação padronizada z-score (x Churn):   ", tenure_z.corr(df["Churn"]))
print("Correlação escalonada min-max (x Churn):    ", tenure_minmax.corr(df["Churn"]))

**Resposta:** O coeficiente de correlação **não muda** (permanece exatamente `-0,352229` nos três casos). Matematicamente, tanto a padronização z-score ($X' = \frac{X - \mu_X}{\sigma_X}$) quanto o escalonamento min-max ($X' = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$) são transformações lineares afins da forma $X' = aX + b$ com fator de escala estritamente positivo ($a > 0$, pois o desvio-padrão e a amplitude são positivos). Pela linearidade da esperança, a nova média é $\bar{X}' = a\bar{X} + b$, de modo que os desvios em relação à média tornam-se $X'_i - \bar{X}' = a(X_i - \bar{X})$. Substituindo na fórmula da correlação de Pearson entre $X'$ e $Y$ (`Churn`):

$$r_{X', Y} = \frac{\sum (X'_i - \bar{X}')(Y_i - \bar{Y})}{\sqrt{\sum (X'_i - \bar{X}')^2}\sqrt{\sum (Y_i - \bar{Y})^2}} = \frac{a \sum (X_i - \bar{X})(Y_i - \bar{Y})}{|a| \sqrt{\sum (X_i - \bar{X})^2}\sqrt{\sum (Y_i - \bar{Y})^2}} = r_{X, Y}$$

Como $a > 0$, tem-se $|a| = a$, e a constante $a$ se cancela no numerador e no denominador, enquanto a constante aditiva $b$ se cancela na subtração da média, mantendo a correlação inalterada.